## Gemini

Push VQA to Google Gemini, *specific for the qualitative analysis*.

### Docs:

* in theory, there is documentation here: https://ai.google.dev/gemini-api/docs, 
* Here though, I just asked claude again :D

#### Key Points (cp-claude)

* API Key: Get your API key from the [Google Console](https://aistudio.google.com/app/apikey) -- note that you either have to create a new GCP project OR create the API key in an existing project
* Then the rest of the steps is similar for Claude/ChatGPT


First, install anthropic api (also, see .yml file for the environment for this project)

In [1]:
#!pip install google-genai

Where are things stored/going to be stored?

What is thinking budget:

| Model | Min | Max | Can disable (=0)? |
|---|---|---|---|
| `gemini-2.5-flash` | 1 | 24,576 | ✅ yes |
| `gemini-2.5-flash-lite` | 512 | 24,576 | ✅ yes |
| `gemini-2.5-pro` | 128 | 32,768 | ❌ no |

Setting `thinking_budget` to `-1` turns on dynamic thinking, where the model automatically adjusts the budget based on the complexity of the request. Google AI When dynamic thinking is enabled this way, the default max is 8,192 tokens.

In [88]:
dir_api = '/Users/jnaiman/Dropbox/jcdl_followup/LMM_outputs/gemini_qual/' #store API results for example data
mds_dir = '/Users/jnaiman/Dropbox/jcdl_followup/LMM_outputs/gemini_qual/markdown/' # where images are stored
model_name='gemini-3.1-pro-preview' 

# how many multi-panel plots to pass through and what number of single panel plots?
n_single = 2
n_multi = 8

key_file = '/Users/jnaiman/.gemini/key.txt'

# where are there images to save/use?
imgs_dir = '/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures_qual/' # where images are stored


# where are the jsons of the original panels?
jsons_dir = '/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/qa_jsons/' # directory where jsons created with figure are stored
imgs_orig_dir = '/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/imgs/' # directory where jsons created with figure are stored

# for saving temp images for reading in
tmp_dir = '/Users/jnaiman/Downloads/tmp/' # this might not be used...

img_format = 'jpeg'

# for asking for reasoning
reasoning_text = 'In addition to providing your answer, please provide your reasoning.'

In [62]:
import base64
from PIL import Image
import numpy as np
import json
import re
import pickle
import pandas as pd
import os
from glob import glob
# import google.generativeai as genai
from google import genai
from google.genai import types

# debug
from importlib import reload
from copy import deepcopy
from shutil import copyfile

from sys import path
path.append('../')
import utils.llm_utils
reload(utils.llm_utils)
from utils.llm_utils import parse_qa, load_image, get_img_json_pair, parse_for_errors

import time
from utils.plot_qa_utils import get_nplots

In [63]:
# setup
with open(key_file,'r') as f:
    api_key = f.read()

# Configure the API key
#genai.configure(api_key=api_key.strip())
client = genai.Client(api_key=api_key.strip())

config=types.GenerateContentConfig(
    #thinking_config=types.ThinkingConfig(thinking_budget=thinking_budget)  # disable for speed/cost
    thinking_config=types.ThinkingConfig()  # disable for speed/cost
)

In [64]:
jsons_to_parse = glob(jsons_dir + '/*.json')
jsons_to_parse[:3]

['/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/qa_jsons/Picture_000015_qa.json',
 '/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/qa_jsons/Picture_000005_qa.json',
 '/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/qa_jsons/Picture_000077_qa.json']

In [65]:
fnames = []
npanels = []

# get images
for j in jsons_to_parse:
    with open(j,'r') as f:
        data = json.load(f)
        data = json.loads(data)
    fnames.append(j.split('/')[-1].removesuffix('_qa.json'))
    npanels.append(data['figure']['nrows']*data['figure']['ncols'])

dfpages = pd.DataFrame({'id':fnames, 'n-panels':npanels})

In [66]:
# get ones already there:
nocopy = False
imgs_synth = glob(imgs_dir + '*')
if len(imgs_synth) > 0:
    #print('not implmented yet!')
    nocopy = True

if nocopy:
    print('Will not copy anything over, delete all if you want something different.')

Will not copy anything over, delete all if you want something different.


In [73]:
if not nocopy:
    dfsingle = dfpages[dfpages['n-panels']==1].sample(n=n_single)
    #dfsingle
    n_panels = dfpages['n-panels'].unique()
    n_panels = np.sort(n_panels)[1:-1] # drop lowest and highest

    n_panels_choose = np.random.choice(n_panels, size=n_multi) # weight equally

    for i,n in enumerate(n_panels_choose):
        d = dfpages[dfpages['n-panels']==n].sample(n=1)
        if i == 0:
            dfmulti = d.copy()
        else:
            dfmulti = pd.concat([dfmulti.copy(),d.copy()])

    dfboth = pd.concat([dfsingle.copy(), dfmulti.copy()]).reset_index()

    for i,r in dfboth.iterrows():
        copyfile(imgs_orig_dir + '/' + r['id'] + '.' + img_format, imgs_dir + '/' + r['id'] + '.' + img_format)
else:
    ffiles = glob(imgs_dir + '/*.' + img_format)
    # construct df
    names = []; npanels = []; jnames=[]
    for f in ffiles:
        names.append(f.split('/')[-1].removesuffix('.'+img_format))

        j = jsons_dir + f.split('/')[-1].removesuffix('.'+img_format) + '_qa.json'
        with open(j,'r') as f:
            data = json.load(f)
            data = json.loads(data)
        npanels.append(data['figure']['nrows']*data['figure']['ncols'])
        jnames.append(j)

    dfboth = pd.DataFrame({'id':names, 'n-panels':npanels, 'jsons_to_parse':jnames})


In [74]:
dfboth

,id,n-panels,jsons_to_parse
0,Picture_000078,2,/Users/jnaiman/Dropbox/jcdl_followup/synthetic...
1,Picture_000023,12,/Users/jnaiman/Dropbox/jcdl_followup/synthetic...
2,Picture_000059,8,/Users/jnaiman/Dropbox/jcdl_followup/synthetic...
3,Picture_000171,3,/Users/jnaiman/Dropbox/jcdl_followup/synthetic...
4,Picture_000080,15,/Users/jnaiman/Dropbox/jcdl_followup/synthetic...
5,Picture_000068,6,/Users/jnaiman/Dropbox/jcdl_followup/synthetic...
6,Picture_000046,1,/Users/jnaiman/Dropbox/jcdl_followup/synthetic...
7,Picture_000093,5,/Users/jnaiman/Dropbox/jcdl_followup/synthetic...
8,Picture_000118,9,/Users/jnaiman/Dropbox/jcdl_followup/synthetic...
9,Picture_000125,1,/Users/jnaiman/Dropbox/jcdl_followup/synthetic...


In [68]:
def send_to_gemini_qual(question_list, image_path, client,model_name='gemini-3.1-flash-lite-preview',
                    test_run = True, 
                    verbose=True, verbose_tokens = False,
                    system_prompt = None, img_format='jpg',         
                   large_image= False, config=None, reasoning=None):
    """
    Sends an image + question to Gemini.  Does something different for large file, but might be bad idea.

    system_prompt : set to None to use default from question list
    """
    if system_prompt is None:
        # print('Must import system_prompt!  Stopping...')
        # import sys; sys.exit()
        system_prompt = question_list['persona']
    if img_format.lower() == 'jpg':
        img_format = 'jpeg'

    err = False
    prompt_save = ''; prompt = ''; response = ''
    if not test_run:
        question = question_list['context'] + " " + question_list['question'] + " " + question_list['format']
        if reasoning is not None:
            question += " " + reasoning
        # question = qual_question
        # lowercase the first word, just in case
        question = question.lstrip() # no whitespace
        question = question[0].lower() + question[1:]

        if verbose: print('   on question:',question)
        # Prepare the API request
        prompt = f"{question}"
        prompt_save = f"{question}"
        try:
            with open(image_path, 'rb') as f:
                image_bytes = f.read()
            image_part = types.Part.from_bytes(data=image_bytes, mime_type='image/'+img_format)

            if config is None:
                config = types.GenerateContentConfig(
                    system_instruction=system_prompt,
                    # other config here
                )
            # Generate content with the uploaded file
            response = client.models.generate_content(
                model=model_name,
                contents=[prompt, image_part],
                config=config
            )
            
            if large_image:
                # Clean up - delete the uploaded file
                client.files.delete(name=uploaded_file.name)
            
        
        except Exception as e:
            print('[ERROR]:', str(e))
            err = True        

    if not test_run and not err:
        if response is None:
            print('[ERROR]: response is None')
            err = True
            text = ''
        else:
            text = ''
            for part in response.candidates[0].content.parts:
                if not part.thought:
                    text = part.text
                    break  # take first non-thought part
            if not text:
                print('[WARNING]: no non-thought parts found in response')
        if verbose:
            print('     response:', text)

        question_list['Response (raw)'] = deepcopy(response)
        # Get the response from the API
        answer = text
        question_list['raw answer'] = answer
        # also calculate usage
        usage = {
            'input_tokens': response.usage_metadata.prompt_token_count,
            'output_tokens': response.usage_metadata.candidates_token_count,
            'total_tokens': response.usage_metadata.total_token_count,
            'cached_content_tokens': getattr(response.usage_metadata, 'cached_content_token_count', 0)
            }

        question_list['usage'] = usage
        if verbose and verbose_tokens:
            print(f"      - Input tokens: {usage.input_tokens}")
            print(f"      - Output tokens: {usage.output_tokens}")
            print(f"      - Total tokens: {usage.input_tokens + usage.output_tokens}")
        # # format answer
        # if '```json"' in answer:
        #     answer_format = answer.split('```json"')[-1].split('\n')[0].replace('\n', '')
        # elif '```json\n' in answer:
        #     answer_format = answer.split('```json\n')[-1].split('\n')[0].replace('\n', '')
        # elif '```json' in answer:
        #     answer_format = answer.split('```json')[-1].split('\n')[0].replace('\n', '')
        # else:
        #     answer_format = answer # just give it a shot!
        # #answer.replace("```json\n",'').replace("\n```",'')
        # try:
        #     question_list['Response'] = json.loads(answer_format)
        # except:
        #     question_list['Response'] = answer_format
        #     question_list['Error'] = 'JSON formatting'
        # question_list['Response String'] = answer_format
    elif err:
        question_list['raw answer'] = 'ERROR'
    else:
        question_list['Response'] = 'TEST RUN'
        question_list['Response String'] = 'TEST RUN'
        question_list['Response (raw)'] = 'TEST RUN'
    

    return question_list, prompt_save, system_prompt

In [69]:
qualitative_qa = {
    "persona": (
        "You are a careful scientific reader describing a figure for a colleague "
        "who cannot see it. You ground every claim in the visual content and do "
        "not speculate beyond what is shown."
    ),
    "context": (
        "The image is a single scientific figure that may contain one or more "
        "panels (subplots) arranged in a grid. Treat axis labels, legends, "
        "titles, tick values, and annotations as authoritative."
    ),
    "question": (
        "Describe this figure in as much detail as possible. At a minimum, cover: (1) what kind of plot each panel is "
        "(e.g., scatter, line, histogram, image/heatmap), (2) what the axes "
        "represent including units when shown, (3) the main visual trend or "
        "finding in each panel, and (4) one or two sentences synthesizing what "
        "the figure as a whole communicates. If the figure has a single panel, "
        "treat it as one panel. Quote labels and legend entries verbatim when "
        "you reference them."
    ),
    "format": (
        ""
    ),
    "reasoning": (
        reasoning_text
        # reuse the existing reasoning_text from the notebook; appends a separate
        # {"explanation":""} JSON snippet so we keep the existing two-blob parse path
    ),
}

In [70]:
qualitative_qa

{'persona': 'You are a careful scientific reader describing a figure for a colleague who cannot see it. You ground every claim in the visual content and do not speculate beyond what is shown.',
 'context': 'The image is a single scientific figure that may contain one or more panels (subplots) arranged in a grid. Treat axis labels, legends, titles, tick values, and annotations as authoritative.',
 'question': 'Describe this figure in as much detail as possible. At a minimum, cover: (1) what kind of plot each panel is (e.g., scatter, line, histogram, image/heatmap), (2) what the axes represent including units when shown, (3) the main visual trend or finding in each panel, and (4) one or two sentences synthesizing what the figure as a whole communicates. If the figure has a single panel, treat it as one panel. Quote labels and legend entries verbatim when you reference them.',
 'format': '',
 'reasoning': 'In addition to providing your answer, please provide your reasoning.'}

In [102]:
reload(utils.llm_utils)
from utils.llm_utils import parse_for_errors

iMax = 20
verbose = True
test_run = False # run w/o actually pinging openai
restart = False
# set system_prompt to None to default to what is in question list
# system_prompt = """You are a helpful assistant that responds only in valid JSON format. Do not include any explanations, reasoning, or text outside of the JSON response."""
# if reasoning_text is not None: # rephrease is reasoning is requested
#     system_prompt = """You are a helpful assistant that responds only in valid JSON format. Do not include any text outside of the requested JSON responses."""

#system_prompt = """You must respond with only valid JSON. Start your response immediately with { and end with }. Do not write any text before or after the JSON."""
# temperature=0.1

fac = 1.0

question_list = deepcopy(qualitative_qa)

for ijson,json_path in enumerate(dfboth['jsons_to_parse'].values):
    if ijson >= iMax:
        continue

    print('on', ijson, 'of', min(iMax,len(dfboth['jsons_to_parse'].values)))

    # get image and base json
    img_path = imgs_dir + json_path.split('/')[-1].removesuffix('_qa.json') + '.' + img_format
    _, img_format_media, base_json, err = get_img_json_pair(img_path, json_path, 
                                                            dir_api, restart=restart,fac=fac,
                                                      tmp_dir=tmp_dir, 
                                                      img_format=img_format) #, load_image=False)
    if err:
        continue
    if verbose: print('Got image!')

    ###### create QA ########
    # qa = []
    
    # # start with figure level questions
    # for k,v in base_json['VQA']['Level 1']['Figure-level questions'].items():
    #     out = {'Q':v['Q'], 'A':v['A'], 'Level':'Level 1', 'type':'Figure-level questions', 'Response':"", 
    #            "persona":v['persona'], 'context':v['context'], 'question':v['question'], 'format':v['format'],
    #            "reasoning":reasoning_text}
    #     qa.append(out)
    # for level in ['Level 2', 'Level 3']:
    #     if level in base_json['VQA']:
    #         if 'Figure-level questions' in base_json['VQA'][level]:
    #             #print('** yes, level ***', level)
    #             for k,v in base_json['VQA'][level]['Figure-level questions'].items():
    #                 out = {'Q':v['Q'], 'A':v['A'], 'Level':level, 'type':'Figure-level questions', 'Response':"", 
    #                     "persona":v['persona'], 'context':v['context'], 'question':v['question'], 'format':v['format'],
    #                     "reasoning":reasoning_text}
    #                 qa.append(out)
    
    # what kinds?
    #types = ['(words + list)', '(words)']
    # types_qa = []
    
    # # get uniques
    # level_parse = 'Level 1'
    # plot_level = 'Plot-level questions'
    # qa = parse_qa(level_parse, plot_level, qa, base_json['VQA'], types_qa, use_split_keys=False)
    
    # level_parse = 'Level 2'
    # plot_level = 'Plot-level questions'
    # qa = parse_qa(level_parse, plot_level, qa, base_json['VQA'], types_qa, use_split_keys=False)
    
    # level_parse = 'Level 3'
    # plot_level = 'Plot-level questions'
    # qa = parse_qa(level_parse, plot_level, qa, base_json['VQA'], types_qa, use_split_keys=False)

    responses = []; prompts = []; system_prompts = []
    # for question_list in qa:
    response, prompt, system_prompt_out = send_to_gemini_qual(question_list, img_path, client,
                test_run = test_run, 
                verbose=verbose, img_format=img_format,
                config=config, reasoning = reasoning_text)
    # responses.append(response)
    question_list['prompt'] = prompt
    question_list['system prompt'] = system_prompt_out
        #import sys; sys.exit()


    # # parse for errors
    # print('')
    # print('**** Cleaned QA ****')
    # qa = parse_for_errors(qa, llm='Gemini')
    #qa = parse_for_errors(qa) # might need to do this again

    # dump to file
    if not test_run:
        with open(dir_api + json_path.split('/')[-1].removesuffix('.json')+ '.pickle', 'wb') as ff:
            pickle.dump([response, question_list], ff)
        print('Just saved:', dir_api + json_path.split('/')[-1].removesuffix('.json')+ '.pickle')
        # also print markdown
        with open(mds_dir + json_path.split('/')[-1].removesuffix('.json') + '.md','w') as f:
            print(question_in['raw answer'], file=f)
    else:
        print('Would store at:', dir_api + json_path.split('/')[-1].removesuffix('.json')+ '.pickle')
    #import sys; sys.exit()

print('!!!!!!!! DONE !!!!!!!!!!')

on 0 of 10
have file already: /Users/jnaiman/Dropbox/jcdl_followup/LMM_outputs/gemini_qual/Picture_000078_qa.pickle
on 1 of 10
have file already: /Users/jnaiman/Dropbox/jcdl_followup/LMM_outputs/gemini_qual/Picture_000023_qa.pickle
on 2 of 10
Got image!
   on question: the image is a single scientific figure that may contain one or more panels (subplots) arranged in a grid. Treat axis labels, legends, titles, tick values, and annotations as authoritative. Describe this figure in as much detail as possible. At a minimum, cover: (1) what kind of plot each panel is (e.g., scatter, line, histogram, image/heatmap), (2) what the axes represent including units when shown, (3) the main visual trend or finding in each panel, and (4) one or two sentences synthesizing what the figure as a whole communicates. If the figure has a single panel, treat it as one panel. Quote labels and legend entries verbatim when you reference them.  In addition to providing your answer, please provide your reasoning

In [ ]:
#question

## Look at data

Check out one, if you wanna:

In [95]:
pickles = glob(dir_api + '*.pickle')
#pickles = glob('/Users/jnaiman/Downloads/tmp/JCDL2025/example_hists/claude_api/*pickle')
pickles[:5]

['/Users/jnaiman/Dropbox/jcdl_followup/LMM_outputs/gemini_qual/Picture_000023_qa.pickle',
 '/Users/jnaiman/Dropbox/jcdl_followup/LMM_outputs/gemini_qual/Picture_000078_qa.pickle']

In [98]:
ifile = 0
with open(pickles[ifile], 'rb') as f:
    response_in, question_in = pickle.load(f)

In [99]:
response_in

{'persona': 'You are a careful scientific reader describing a figure for a colleague who cannot see it. You ground every claim in the visual content and do not speculate beyond what is shown.',
 'context': 'The image is a single scientific figure that may contain one or more panels (subplots) arranged in a grid. Treat axis labels, legends, titles, tick values, and annotations as authoritative.',
 'question': 'Describe this figure in as much detail as possible. At a minimum, cover: (1) what kind of plot each panel is (e.g., scatter, line, histogram, image/heatmap), (2) what the axes represent including units when shown, (3) the main visual trend or finding in each panel, and (4) one or two sentences synthesizing what the figure as a whole communicates. If the figure has a single panel, treat it as one panel. Quote labels and legend entries verbatim when you reference them.',
 'format': '',
 'reasoning': 'In addition to providing your answer, please provide your reasoning.',
 'Response (

In [ ]:
# with open(mds_dir + pickles[ifile].split('/')[-1].removesuffix('.pickle') + '.md','w') as f:
#     print(question_in['raw answer'], file=f)

In [101]:
print(mds_dir + pickles[ifile].split('/')[-1].removesuffix('.pickle') + '.md')

/Users/jnaiman/Dropbox/jcdl_followup/LMM_outputs/gemini_qual/markdown/Picture_000023_qa.md
